# 네이버 뉴스 본문 수집 — 검색어 방식

앞 단계(`검색어_뉴스_url_수집.ipynb`)에서 만든 `링크_{검색어}_{기간}.json`을 읽어, 기사마다 **제목·본문·날짜·카테고리**를 모아 CSV로 저장하는 노트북임.

**쓰는 순서**
1. 아래 `설정` 셀의 `query_ranges`를 앞 단계와 **똑같이** 맞추기
2. 위에서부터 셀 순서대로 실행
3. `본문_{검색어}_{기간}.csv`가 생기면 끝. 맨 아래 통합 셀을 돌리면 검색어별 파일을 하나로 합침

- 입력: `링크_{검색어}_{YYMMDD}_{YYMMDD}.json`
- 출력: `본문_{검색어}_{YYMMDD}_{YYMMDD}.csv` (열: link, pubdate, category, title, body, query)
- 보조 파일: 중간 저장 JSON, 두 번 시도해도 실패한 주소 JSON
- 기본은 **BS4**(빠름). 설정에서 크롬 창(Selenium)으로 바꿀 수 있고, 실패한 기사만 크롬으로 다시 시도하는 것도 됨
- 중간에 끊겨도 중간 저장 파일이 있으면 이어서 수집

> 개인 학습·연구용으로 쓰는 걸 전제로 함. 대기 시간을 줄이거나 여러 개를 동시에 돌리는 식으로 서버에 부담 주지 말 것


In [ ]:
# 필요한 도구 설치 — 이미 깔려 있으면 건너뛰어도 됨
# %pip install -q requests beautifulsoup4 pandas selenium

# Colab에서 크롬 창 방식(Selenium)을 쓸 때만 — 크롬이 없어서 따로 설치해야 함
# !apt-get -qq update && apt-get -qq install -y chromium-browser chromium-chromedriver

In [ ]:
# ============================== 설정 ==============================
# 여기만 고치면 됨. 앞 단계(url 수집)와 검색어·기간을 똑같이 맞출 것
# =================================================================

# 검색어와 기간 — 앞 단계에서 쓴 것과 동일하게
# 날짜 형식: 'YYYY.MM.DD'
query_ranges = [
    {'query': '기후변화', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    # {'query': '전기차 보조금', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
]

# 앞 단계에서 True로 뒀으면 여기도 True (파일이 달마다 나뉘어 있어서 이름이 달라짐)
SPLIT_BY_MONTH = False

# 본문을 가져오는 방식
#   'bs4'      — 주소만 읽어서 본문을 뽑음. 빠름 (기사당 1초 안팎), 기본값
#   'selenium' — 크롬 창으로 기사를 하나씩 염. 눈으로 확인되지만 느림 (기사당 2~3초)
ENGINE = 'bs4'

# BS4로 두 번 시도해도 실패한 기사만 크롬 창으로 다시 열어 볼지 여부
# 자바스크립트로 늦게 뜨는 기사 몇 건을 건지는 용도. 크롬이 없으면 알아서 건너뜀
RETRY_WITH_SELENIUM = True

# 크롬 창을 보면서 돌리려면 False, 창 없이 돌리려면 True (Colab은 자동으로 True)
HEADLESS = False

# 저장 폴더 — 결과 파일이 전부 여기에 쌓임
# 직접 정하려면 경로 문자열 입력 (예: '/home/me/뉴스수집', Colab이면 '/content/drive/MyDrive/내프로젝트/data')
# None이면 자동 — 로컬은 프로젝트 폴더 아래 data/뉴스수집, Colab은 드라이브의 MyDrive/data/뉴스수집
# 드라이브에 따로 만들어 둔 프로젝트 폴더 안에 넣고 싶으면 그 경로를 직접 적을 것
# 앞 단계(url 수집)와 같은 폴더여야 링크 파일을 찾음
SAVE_DIR_OVERRIDE = None

# Colab에서 구글 드라이브에 저장할지 여부
MOUNT_DRIVE = True

# 최종 CSV가 이미 있는 작업은 다시 수집하지 않음
SKIP_COMPLETED = True

# 몇 건마다 중간 저장할지 — 중간에 끊겨도 여기까지는 남음
CHECKPOINT_INTERVAL = 100

# 서버에 부담 주지 않도록 쉬는 시간(초)
ARTICLE_PAUSE_RANGE_SEC = (0.4, 1.2)  # 기사 하나 끝내고 다음 기사로 넘어갈 때
JOB_PAUSE_RANGE_SEC = (8, 20)         # 검색어(작업) 하나 끝내고 다음 작업으로 넘어갈 때
REQUEST_TIMEOUT_SEC = 10              # 응답이 이만큼 없으면 그 기사는 실패 처리

In [ ]:
import calendar
import json
import os
import platform
import random
import re
import shutil
import sys
import time
import unicodedata
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

# 지금 Colab에서 돌고 있는지 확인 — 경로와 창 설정이 달라짐
IN_COLAB = 'google.colab' in sys.modules

# Colab은 화면이 없어서 크롬 창을 띄울 수 없음
if IN_COLAB and not HEADLESS:
    HEADLESS = True
    print('Colab이라 창 없이 실행함 (HEADLESS=True)')

# Colab이면 드라이브 연결 — 앞 단계에서 만든 링크 파일도 여기서 읽어 옴
if IN_COLAB and MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'드라이브 연결 실패 — 세션 안에만 저장됨: {exc!r}')


# 저장 폴더의 기준이 될 위치 찾기 (앞 단계 노트북과 같은 규칙)
# Colab은 드라이브 최상단(MyDrive), 로컬은 프로젝트 최상위 — 다른 곳에 넣으려면 위 SAVE_DIR_OVERRIDE 사용
def detect_project_dir():
    if IN_COLAB:
        drive_root = Path('/content/drive/MyDrive')
        return drive_root if drive_root.exists() else Path('/content')

    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / '.git').exists() or (candidate / 'pipeline_py').exists():
            return candidate
    return here


PROJECT_DIR = detect_project_dir()

# 저장 폴더 — 앞 단계가 만든 링크 파일을 여기서 읽고, 본문 CSV도 여기에 저장
SAVE_DIR = Path(SAVE_DIR_OVERRIDE) if SAVE_DIR_OVERRIDE else PROJECT_DIR / 'data' / '뉴스수집'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 기사 페이지를 읽을 때 쓰는 기본 정보 — 앞 단계와 같은 값
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'


# 검색어를 파일 이름으로 쓸 수 있게 정리 (앞 단계와 같은 규칙이어야 파일을 찾음)
def safe_name(text):
    name = unicodedata.normalize('NFC', str(text)).strip()
    name = re.sub(r'\s+', '_', name)
    name = re.sub(r'[\\/:*?"<>|]', '_', name)
    return name or 'query'


# 파일 이름에 붙는 기간 표시 만들기 (2026.05.05 ~ 2026.05.11 -> '260505_260511')
def make_period_suffix(start_date, end_date):
    return f"{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}"


# 설정한 검색어·기간을 실제로 돌릴 작업 목록으로 바꿈 (앞 단계와 같은 규칙)
def build_jobs(query_ranges, split_by_month=SPLIT_BY_MONTH):
    jobs = []
    for item in query_ranges:
        query = item['query']
        start = datetime.strptime(item['start_date'], '%Y.%m.%d')
        end = datetime.strptime(item['end_date'], '%Y.%m.%d')
        if start > end:
            raise ValueError(f'시작일이 종료일보다 늦습니다: {item}')

        if not split_by_month:
            jobs.append({
                'query': query,
                'start_date': start.strftime('%Y.%m.%d'),
                'end_date': end.strftime('%Y.%m.%d'),
            })
            continue

        current = start
        while current <= end:
            last_day = calendar.monthrange(current.year, current.month)[1]
            month_end = current.replace(day=last_day)
            chunk_end = min(month_end, end)
            jobs.append({
                'query': query,
                'start_date': current.strftime('%Y.%m.%d'),
                'end_date': chunk_end.strftime('%Y.%m.%d'),
            })
            current = month_end + timedelta(days=1)
    return jobs


# 한글 파일 이름이 컴퓨터마다 다르게 저장될 수 있어(맥/윈도우/드라이브) 이름을 정리해서 비교
# 같은 이름의 파일이 없으면 새로 저장할 경로를 돌려줌
def resolve_nfc_path(save_dir, filename):
    target = unicodedata.normalize('NFC', filename)
    direct = save_dir / filename
    if direct.exists():
        return direct
    for cand in save_dir.iterdir():
        if unicodedata.normalize('NFC', cand.name) == target:
            return cand
    return save_dir / target


jobs = build_jobs(query_ranges)

# 기사 페이지를 열 때마다 같은 접속 정보와 한국어 설정 사용
session = requests.Session()
session.headers.update({
    'User-Agent': USER_AGENT,
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
})

print(f'실행 환경: {"Colab" if IN_COLAB else "로컬"}')
print(f'저장 폴더: {SAVE_DIR}')
print(f'본문 가져오는 방식: {ENGINE}' + (' (실패분은 크롬 창으로 재시도)' if RETRY_WITH_SELENIUM and ENGINE == "bs4" else ''))
print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    name = safe_name(job['query'])
    period = make_period_suffix(job['start_date'], job['end_date'])
    print(f'   {job}  ->  링크_{name}_{period}.json  ->  본문_{name}_{period}.csv')

In [ ]:
# 랜덤 대기 후 로그 출력 — 매번 똑같은 간격으로 요청하지 않게 함
def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f'{label} {pause_sec:.1f}초 대기')
    time.sleep(pause_sec)


# 일반 뉴스 주소가 스포츠/연예 페이지로 넘어가는 경우가 있어 따로 구분
# 앞은 주소에 들어 있는 글자, 뒤는 카테고리 이름과 세부 종목을 찾는 규칙
REDIRECT_DOMAINS = {
    'sports.naver.com': ('스포츠', r'https://m\.sports\.naver\.com/([^/]+)/article/'),
    'entertain.naver.com': ('연예', r'https://m\.entertain\.naver\.com/([^/]+)/article/'),
}


# 찾는 항목이 없으면 빈 글자로 돌려줌
def get_text_or_empty(soup, css_rule):
    element = soup.select_one(css_rule)
    return element.get_text(strip=True) if element else ''


# 본문 안의 줄바꿈과 연속 공백을 한 칸으로 정리 — CSV에 넣기 좋게
def normalize_body_text(text):
    return re.sub(r'\s+', ' ', text).strip()


# 화면에 보이는 한글 날짜를 컴퓨터가 읽는 형태로 변경
# '2026.05.05. 오전 7:10' -> '2026-05-05 07:10:00'
def parse_korean_datetime(text):
    m = re.match(r'(\d{4})\.(\d{2})\.(\d{2})\.\s*(오전|오후)\s*(\d{1,2}):(\d{2})', text)
    if not m:
        return ''
    y, mo, d, ampm, h, mi = m.groups()
    h = int(h)
    # 오후는 12를 더하고, 오전 12시는 0시로 (오후 12시는 그대로 12시)
    if ampm == '오후' and h != 12:
        h += 12
    elif ampm == '오전' and h == 12:
        h = 0
    return f'{y}-{mo}-{d} {h:02d}:{mi}:00'


# 주소를 보고 스포츠/연예 카테고리 정리 — 'm.sports.naver.com/golf/article/...' -> '스포츠/golf'
def detect_redirect_category(current_url):
    for domain, (prefix, pattern) in REDIRECT_DOMAINS.items():
        if domain in current_url:
            cat_match = re.match(pattern, current_url)
            return f'{prefix}/{cat_match.group(1)}' if cat_match else prefix
    return '기타'


# 스포츠/연예 기사는 일반 뉴스와 페이지 구조가 달라 따로 추출 (BS4)
def extract_redirected_article_bs4(response, original_link):
    soup = BeautifulSoup(response.text, 'html.parser')

    # 제목 — 페이지 정보에 들어 있는 제목 사용
    title_meta = soup.find('meta', property='og:title')
    title = title_meta.get('content', '').strip() if title_meta else ''

    # 본문 — 스포츠/연예 기사 본문 영역
    body_el = soup.select_one('div._article_content')
    body = normalize_body_text(body_el.get_text(' ', strip=True)) if body_el else ''

    # 날짜 — em.date 첫 번째가 입력일, 두 번째는 수정일이라 안 씀
    em = soup.select_one('em.date')
    pubdate = parse_korean_datetime(em.get_text(strip=True)) if em else ''

    # 셋 중 하나라도 비면 실패로 두고 나중에 다시 시도
    if not title or not body or not pubdate:
        raise ValueError(f'스포츠/연예 기사: title={bool(title)}, body={bool(body)}, pubdate={bool(pubdate)}')

    # 저장할 때는 처음에 모아 둔 기사 주소 그대로 남김
    return {
        'link': original_link,
        'pubdate': pubdate,
        'category': detect_redirect_category(response.url),
        'title': title,
        'body': body,
    }


# 기사 한 건에서 제목·본문·날짜·카테고리 가져오기 (BS4)
# 네이버 뉴스는 주소만 읽어도 본문이 들어 있어서 크롬 없이도 됨
def extract_article_bs4(link):
    response = session.get(link, timeout=REQUEST_TIMEOUT_SEC)
    response.raise_for_status()

    # 스포츠/연예로 넘어간 기사면 위에서 만든 함수로 처리
    if any(domain in response.url for domain in REDIRECT_DOMAINS):
        return extract_redirected_article_bs4(response, link)

    soup = BeautifulSoup(response.text, 'html.parser')

    title = get_text_or_empty(soup, '.media_end_head_headline')

    # 본문은 문단마다 태그가 나뉘어 있어서 그냥 붙이면 '~했다.정부는'처럼 문장이 달라붙음
    # 태그 사이에 공백을 한 칸씩 넣어 읽은 뒤 연속 공백을 정리 — 크롬 창 방식 결과와도 모양이 같아짐
    body_el = soup.select_one('#newsct_article')
    body = normalize_body_text(body_el.get_text(' ', strip=True)) if body_el else ''

    # 날짜는 화면 글자 대신 페이지 안에 들어 있는 날짜값 사용
    pubdate_element = soup.select_one('span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')
    pubdate = pubdate_element.get('data-date-time', '') if pubdate_element else ''

    # 카테고리는 위쪽 탭 중 선택돼 있는 항목
    category = get_text_or_empty(soup, 'a.Nitem_link[aria-selected="true"] span.Nitem_link_menu')

    if not title or not body or not pubdate:
        raise ValueError('title/body/pubdate 중 일부를 추출하지 못함')

    return {
        'link': link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }

In [ ]:
# ===== 크롬 창(Selenium) 방식 — ENGINE='selenium'이거나 실패분 재시도에서만 사용 =====
# 크롬 창은 실제로 필요할 때 처음 한 번만 띄움

_driver = None  # 띄워 둔 크롬 창을 담아 두는 자리


# 이 컴퓨터에 깔린 크롬(또는 크로미움) 위치 찾기
def find_chrome_binary():
    candidates = []

    if platform.system() == 'Windows':
        candidates.extend([
            os.path.expandvars(r'%ProgramFiles%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%LocalAppData%\Google\Chrome\Application\chrome.exe'),
        ])
    else:
        for name in ['google-chrome', 'google-chrome-stable', 'chromium-browser', 'chromium']:
            found = shutil.which(name)
            if found:
                candidates.append(found)
        candidates.append('/Applications/Google Chrome.app/Contents/MacOS/Google Chrome')

    for path in candidates:
        if path and Path(path).exists():
            return str(Path(path))
    return None


# 크롬 창 띄우기 — 이미 띄워 뒀으면 그걸 그대로 씀
# selenium이 안 깔려 있거나 크롬이 없으면 오류를 그대로 올려서 부르는 쪽에서 건너뛰게 함
def get_driver():
    global _driver
    if _driver is not None:
        return _driver

    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service

    options = Options()
    options.add_argument(f'user-agent={USER_AGENT}')  # 접속 정보를 일정하게 유지
    options.add_argument('--lang=ko-KR')  # 한국어 페이지로 받기
    options.add_experimental_option('excludeSwitches', ['enable-automation'])  # 자동 실행 창처럼 보이는 표시 줄이기
    options.add_experimental_option('useAutomationExtension', False)  # 자동 실행 창처럼 보이는 표시 줄이기
    options.add_argument('--disable-blink-features=AutomationControlled')  # 자동 실행 창처럼 보이는 표시 줄이기
    options.add_argument('--window-size=1400,1000')  # 항상 비슷한 화면 크기로 열기

    if HEADLESS:
        options.add_argument('--headless=new')  # 창 없이 실행

    if platform.system() != 'Windows':
        options.add_argument('--no-sandbox')  # 리눅스/WSL/Colab에서 크롬 실행 오류 줄이기
        options.add_argument('--disable-dev-shm-usage')  # 크롬이 중간에 꺼지는 문제 줄이기
        options.add_argument('--disable-gpu')  # 창 없이 실행할 때 그래픽 관련 오류 줄이기

    chrome_binary = find_chrome_binary()
    if chrome_binary:
        options.binary_location = chrome_binary
        print(f'크롬 위치: {chrome_binary}')
    else:
        print('크롬을 직접 찾지 못해 Selenium 기본 방식으로 실행')

    _driver = webdriver.Chrome(service=Service(), options=options)
    # 네이버가 자동 실행 창이라고 판단할 가능성을 조금 낮춤
    _driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })
    print('크롬 창 준비 완료')
    return _driver


# 띄워 둔 크롬 창 닫기
def close_driver():
    global _driver
    if _driver is not None:
        try:
            _driver.quit()
            print('브라우저 종료 완료')
        except Exception as exc:
            print(f'브라우저 종료 중 오류: {exc!r}')
        _driver = None


# 기사 한 건에서 제목·본문·날짜·카테고리 가져오기 (크롬 창)
# 본문이 늦게 뜨는 기사가 있어서 한 번 더 확인함
def extract_article_selenium(link):
    from selenium.webdriver.common.by import By

    driver = get_driver()
    driver.get(link)
    time.sleep(0.8)  # 페이지가 뜰 시간 조금 주기

    current_url = driver.current_url

    # 스포츠/연예로 넘어간 기사는 제목·본문·날짜 위치가 다름
    if any(domain in current_url for domain in REDIRECT_DOMAINS):
        title_elements = driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:title"]')
        title = title_elements[0].get_attribute('content').strip() if title_elements else ''
        body_elements = driver.find_elements(By.CSS_SELECTOR, 'div._article_content')
        body = normalize_body_text(body_elements[0].text) if body_elements else ''
        date_elements = driver.find_elements(By.CSS_SELECTOR, 'em.date')
        pubdate = parse_korean_datetime(date_elements[0].text.strip()) if date_elements else ''
        category = detect_redirect_category(current_url)

        if not title or not body or not pubdate:
            raise ValueError(f'스포츠/연예 기사: title={bool(title)}, body={bool(body)}, pubdate={bool(pubdate)}')
        return {'link': link, 'pubdate': pubdate, 'category': category, 'title': title, 'body': body}

    # 일반 뉴스 — 두 번까지 확인해서 늦게 뜨는 본문도 잡음
    title, body, pubdate, category = '', '', '', ''
    for attempt in range(2):
        title_elements = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
        body_elements = driver.find_elements(By.ID, 'newsct_article')
        pubdate_elements = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')
        category_elements = driver.find_elements(By.CSS_SELECTOR, 'a.Nitem_link[aria-selected="true"] span.Nitem_link_menu')

        title = title_elements[0].text.strip() if title_elements else title
        body = normalize_body_text(body_elements[0].text) if body_elements else body
        pubdate = pubdate_elements[0].get_attribute('data-date-time') if pubdate_elements else pubdate
        category = category_elements[0].text.strip() if category_elements else category

        if title and body and pubdate:
            break
        if attempt == 0:
            time.sleep(2.0)  # 아직 안 떴으면 조금 더 기다린 뒤 다시 확인

    if not title or not body or not pubdate:
        raise ValueError(f'title/body/pubdate 추출 실패: title={bool(title)}, body={bool(body)}, pubdate={bool(pubdate)}')

    return {'link': link, 'pubdate': pubdate, 'category': category, 'title': title, 'body': body}


# 설정한 방식(ENGINE)에 맞는 추출 함수로 넘겨 줌
def extract_article(link):
    if ENGINE == 'selenium':
        return extract_article_selenium(link)
    return extract_article_bs4(link)

In [ ]:
# 검색어 하나 × 기간 하나를 본문 CSV 1개로 저장
# 중간 저장으로 이어서 수집하고, 실패한 기사는 한 번 더 시도한 뒤 목록으로 남김
def collect_bodies(query, start_date, end_date, save_dir=SAVE_DIR):
    name = safe_name(query)
    period = make_period_suffix(start_date, end_date)

    links_path = resolve_nfc_path(save_dir, f'링크_{name}_{period}.json')
    checkpoint_path = resolve_nfc_path(save_dir, f'중간저장_본문_{name}_{period}.json')
    csv_save_path = resolve_nfc_path(save_dir, f'본문_{name}_{period}.csv')

    # 최종 파일이 이미 있으면 같은 작업은 다시 하지 않음
    if SKIP_COMPLETED and csv_save_path.exists():
        print()
        print(f'=== {query} / {start_date} ~ {end_date} 이미 완료됨, 건너뜀 ===')
        print(f'기존 파일: {csv_save_path}')
        return csv_save_path

    # 앞 단계에서 만든 링크 파일 읽기
    if not links_path.exists():
        raise FileNotFoundError(f'링크 파일이 없음 — 검색어_뉴스_url_수집.ipynb를 먼저 실행하세요\n찾은 경로: {links_path}')

    with links_path.open('r', encoding='utf-8') as f:
        naver_news_links = json.load(f)

    # 대략 얼마나 걸릴지 미리 알려 줌 (인터넷 상태에 따라 달라짐)
    avg_pause_sec = sum(ARTICLE_PAUSE_RANGE_SEC) / 2
    est_sec_per_article = avg_pause_sec + (2.0 if ENGINE == 'selenium' else 0.7)
    print()
    print(f'=== {query} / {start_date} ~ {end_date} 본문 수집 시작 ({ENGINE}) ===')
    print(f'링크 {len(naver_news_links)}개 불러옴: {links_path}')
    print(f'예상 소요 시간: 약 {len(naver_news_links) * est_sec_per_article / 60:.0f}분')

    # 중간에 끊겼던 작업이 있으면 이어받기 — next_i 다음 기사부터 시작
    if checkpoint_path.exists():
        with checkpoint_path.open('r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        # JSON은 번호를 글자로 저장하니 다시 숫자로 변경
        all_results = {int(k): v for k, v in checkpoint.get('all_results', {}).items()}
        err_idx = checkpoint.get('err_idx', [])
        i = checkpoint.get('next_i', 0)
        print(f'중간 저장 발견 — {i}번째부터 이어서 시작 (이미 수집: {len(all_results)}건)')
    else:
        all_results = dict()
        err_idx = []
        i = 0
        print('새로 시작')

    def save_checkpoint():
        with checkpoint_path.open('w', encoding='utf-8') as f:
            json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i}, f, ensure_ascii=False, indent=2)

    # 링크를 하나씩 열어 본문 수집
    for link in naver_news_links[i:]:
        try:
            all_results[i] = extract_article(link)
            print(f'[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% \t 실패 누적: {len(err_idx)}')
            i += 1

            # 정해 둔 건수마다 중간 저장
            if i % CHECKPOINT_INTERVAL == 0:
                save_checkpoint()
                print(f'중간 저장 — {i}건 완료')

            polite_sleep('다음 기사 전', ARTICLE_PAUSE_RANGE_SEC)

        except Exception as exc:
            print(f'오류 발생 — {i}번째: {exc!r}')
            # 실패한 번호는 따로 모아 두고 아래에서 한 번 더 시도
            err_idx.append(i)
            i += 1
            save_checkpoint()

    # 1차 실패분 재시도 — 잠깐 느렸던 것뿐인 경우가 많아서 대개 여기서 건짐
    if err_idx:
        print()
        print(f'실패 {len(err_idx)}건 재시도 시작...')
        re_err_idx = []
        for retry_i in err_idx:
            try:
                all_results[retry_i] = extract_article(naver_news_links[retry_i])
                print(f'재시도 성공 — {retry_i}번째')
                polite_sleep('다음 재시도 전', ARTICLE_PAUSE_RANGE_SEC)
            except Exception as exc:
                print(f'재시도 실패 — {retry_i}번째: {exc!r}')
                re_err_idx.append(retry_i)
        err_idx = re_err_idx
        print(f'재시도 완료 — 남은 실패: {len(err_idx)}건')

    # 2차 실패분을 크롬 창으로 한 번 더 — BS4로 못 읽는 기사를 건지는 단계
    # 크롬이나 selenium이 없으면 그냥 건너뜀
    if err_idx and RETRY_WITH_SELENIUM and ENGINE == 'bs4':
        print()
        print(f'남은 실패 {len(err_idx)}건을 크롬 창으로 재시도...')
        try:
            get_driver()
        except Exception as exc:
            print(f'크롬을 띄우지 못해 건너뜀 (selenium/크롬 설치 확인): {exc!r}')
        else:
            selenium_err_idx = []
            for retry_i in err_idx:
                try:
                    all_results[retry_i] = extract_article_selenium(naver_news_links[retry_i])
                    print(f'크롬 재시도 성공 — {retry_i}번째')
                    polite_sleep('다음 재시도 전', ARTICLE_PAUSE_RANGE_SEC)
                except Exception as exc:
                    print(f'크롬 재시도 실패 — {retry_i}번째: {exc!r}')
                    selenium_err_idx.append(retry_i)
            err_idx = selenium_err_idx
            print(f'크롬 재시도 완료 — 남은 실패: {len(err_idx)}건')

    # 모은 기사들을 표로 변환
    df = pd.DataFrame(all_results).T
    if df.empty:
        raise ValueError('수집된 본문이 하나도 없습니다.')

    # 같은 내용이 두 번 들어간 경우 제거
    df = df.drop_duplicates().reset_index(drop=True)
    # 어떤 검색어로 모은 기사인지 표시 — 나중에 합칠 때 씀
    df['query'] = query
    # 날짜순으로 정렬해서 저장
    df['pubdate'] = pd.to_datetime(df['pubdate'], errors='coerce')
    df = df.sort_values('pubdate').reset_index(drop=True)

    df.to_csv(csv_save_path, index=False, encoding='utf-8-sig')
    print(f'저장 완료: {csv_save_path}')

    if err_idx:
        # 끝까지 실패한 기사 주소는 따로 남겨서 나중에 사람이 확인할 수 있게 함
        failed_path = save_dir / f'재실패_본문_{name}_{period}.json'
        with failed_path.open('w', encoding='utf-8') as f:
            json.dump({'err_idx': err_idx, 'links': [naver_news_links[x] for x in err_idx]}, f, ensure_ascii=False, indent=2)
        print(f'재실패 목록 저장: {failed_path}')
    elif checkpoint_path.exists():
        # 실패가 하나도 없을 때만 중간 저장 파일 삭제
        checkpoint_path.unlink()

    print(f'본문 수집 완료 — 총 {len(df)}건 / 실패 {len(err_idx)}건')
    return csv_save_path


# 작업 목록을 순서대로 실행 — 하나가 실패해도 기록만 남기고 다음 작업으로
results = []
failures = []
for index, job in enumerate(jobs, start=1):
    print()
    print(f'[{index}/{len(jobs)}] 작업 실행: {job}')
    try:
        results.append(collect_bodies(**job))
    except Exception as exc:
        failures.append({'job': job, 'error': repr(exc)})
        print(f'작업 실패, 다음 작업으로 넘어감: {exc!r}')
    finally:
        if index < len(jobs):
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

if failures:
    failures_path = SAVE_DIR / '수집실패목록_본문.json'
    with failures_path.open('w', encoding='utf-8') as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print()
    print(f'실패 작업 {len(failures)}개 저장: {failures_path}')

print()
print('전체 작업 완료')
print(f'성공/건너뜀: {len(results)}개, 실패: {len(failures)}개')
for result_path in results:
    print(result_path)

In [ ]:
# 크롬 창을 띄웠으면 닫기 (BS4만 썼으면 아무 일도 안 일어남)
close_driver()

## 검색어별 본문 CSV 합치기 (선택)

검색어가 여러 개일 때 `본문_{검색어}_{기간}.csv`들을 하나로 합치는 단계임. 검색어가 하나뿐이면 안 해도 됨.

- 입력: 위에서 만든 `본문_*.csv` 중 **이번 설정에 있는 기간**의 파일들
- 출력: `통합_본문_{기간}.csv`
- 같은 기사가 여러 검색어에 걸린 경우 주소 기준으로 한 번만 남김


In [ ]:
# === 검색어별 본문 CSV 합치기 ===
# 이번 설정(query_ranges)에 들어 있는 기간의 파일만 모음 — 폴더에 남은 예전 파일이 섞이지 않게
target_periods = {make_period_suffix(job['start_date'], job['end_date']) for job in jobs}
target_names = {safe_name(job['query']) for job in jobs}
print(f'합칠 기간: {sorted(target_periods)}')

infos = []
for path in SAVE_DIR.iterdir():
    if not path.is_file():
        continue
    m = re.match(r'^본문_(.+)_(\d{6})_(\d{6})\.csv$', unicodedata.normalize('NFC', path.name))
    if not m:
        continue
    name, s, e = m.groups()
    if f'{s}_{e}' not in target_periods or name not in target_names:
        continue
    infos.append((name, s, e, path))

infos.sort(key=lambda x: (x[0], x[1]))
if not infos:
    raise FileNotFoundError(f'합칠 본문_*.csv가 없습니다: {SAVE_DIR}')

frames = []
for name, s, e, path in infos:
    d = pd.read_csv(path, encoding='utf-8-sig')
    # 예전에 만든 파일이라 query 열이 없으면 파일 이름에서 가져옴
    if 'query' not in d.columns:
        d['query'] = name
    d['source_period'] = f'{s}_{e}'
    d['source_file'] = unicodedata.normalize('NFC', path.name)
    frames.append(d)
    print(f'  {name} {s}_{e}: {len(d)}행')

merged = pd.concat(frames, ignore_index=True)

before = len(merged)
# 같은 기사가 여러 검색어에 걸릴 수 있어 주소 기준으로 한 번만 남김
if 'link' in merged.columns:
    merged = merged.drop_duplicates(subset=['link'], keep='first').reset_index(drop=True)
if 'pubdate' in merged.columns:
    merged['pubdate'] = pd.to_datetime(merged['pubdate'], errors='coerce')
    merged = merged.sort_values(['query', 'pubdate']).reset_index(drop=True)

period_suffix = f"{min(s for _, s, _, _ in infos)}_{max(e for _, _, e, _ in infos)}"
combined_path = SAVE_DIR / f'통합_본문_{period_suffix}.csv'
merged.to_csv(combined_path, index=False, encoding='utf-8-sig')

print(f'\n통합 저장 완료: {combined_path}')
print(f'행 수: {before} -> {len(merged)} (중복 {before - len(merged)}건 제거)')
print('검색어별:', merged['query'].value_counts().to_dict())